# NeuralZip: end-to-end walkthrough

This notebook demonstrates NeuralZip's public workflow with an ordinary Hugging Face model:

1. learn a reusable statistical **setup**;
2. encode the model losslessly and inspect its size and timing;
3. reuse or transfer the setup;
4. run inference while selected BF16 weights remain compressed on the GPU;
5. restore and decode an ordinary model, then verify exact reconstruction.

Throughout the notebook, **compression ratio** means compressed bytes divided by raw bytes, so lower is better. Setup, target mapping, hot encoding, GPU preparation, and decoding are reported separately because they happen at different stages.

| Stage | Main call | Result |
|---|---|---|
| Learn reusable structure | `compressor.fit(model)` | Clusters and shared Huffman codebooks |
| Encode one target | `compressor.encode(model, setup=setup)` | An `EncodedModel` with payloads and metadata |
| Prepare GPU execution | `encoded.prepare_inference(model)` | A model backed by compressed GPU weights |
| Reconstruct | `compressor.decode(encoded)` | A fresh ordinary PyTorch/Hugging Face model |

## 1. Requirements and imports

The CPU compression example requires NeuralZip:

```bash
python -m pip install -e .
```

The GPU section additionally requires CUDA 12, an NVIDIA GPU with BF16 support, and the GPU extra:

```bash
python -m pip install -e ".[gpu,experiments]"
```

In [ ]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, pipeline

from neuralzip import Compressor

## 2. Load a BF16 model

NeuralZip accepts an ordinary `torch.nn.Module`. We use a small causal language model to keep the walkthrough practical. BF16 is also the format supported by the current compressed GPU inference runtime.

In [ ]:
model_id = "Qwen/Qwen2.5-0.5B"

# Load an ordinary Hugging Face model in inference mode.
model = AutoModelForCausalLM.from_pretrained(
    model_id,
    dtype=torch.bfloat16,
).eval()
tokenizer = AutoTokenizer.from_pretrained(model_id)

## 3. Configure the compressor

NeuralZip exposes a model-level `encode`/`decode` interface and learns a reusable setup.

- `ngram_values=()` selects **sharing-only** NeuralZip: exponent streams use precomputed shared Huffman codebooks without packed exponent n-grams. This is the mode currently supported by GPU inference.
- `EncodedModel` always retains the payloads needed by `decode()` and `prepare_inference()`, so `keep_payloads=True` is unnecessary for this API.

In [ ]:
# Sharing-only NeuralZip supports both storage and compressed GPU inference.
neuralzip = Compressor(
    "neuralzip",
    ngram_values=(),
)

## 4. Learn a reusable setup

`fit()` analyzes exponent distributions, clusters compatible tensor streams with Jensen–Shannon distance, and builds shared Huffman codebooks. **It does not encode the model weights.**

The returned setup is an independent data value. Its cost can be paid once and amortized across repeated checkpoints or transferred to a compatible model family member.

In [ ]:
# Learn clusters and Huffman codebooks; no checkpoint is encoded yet.
setup = neuralzip.fit(model)
setup_report = setup.setup_report

print(
    f"Setup: {setup_report.setup_seconds:.4f} s "
    f"(clustering: {setup_report.clustering_seconds:.4f} s, "
    f"codebooks: {setup_report.codebook_setup_seconds:.4f} s)"
)

## 5. Encode the model

`encode()` consumes the learned setup and returns an `EncodedModel`. This value contains the compressed payloads, shared codebooks, reconstruction metadata, byte counts, and no reference to the original weight tensors.

With `return_stats=True`, encoding also returns a compact statistics object:

- `raw_bytes` and `compressed_bytes` describe the complete encoded tensor representation;
- `compression_ratio = compressed_bytes / raw_bytes`;
- `mapping_seconds` measures target-to-setup matching when needed;
- `encode_seconds` measures the hot, target-dependent encoding path;
- `total_seconds` is mapping plus encoding for this call and deliberately excludes the reusable `fit()` cost.

In [ ]:
# NeuralZip reuses the setup; fit() is not repeated.
compressed_model_neuralzip, stats_neuralzip = neuralzip.encode(
    model,
    setup=setup,
    return_stats=True,
)

In [ ]:
# Report hot encoding time and total encoded size. Lower ratios are better.
print(
    f"NeuralZip — encode: {stats_neuralzip.encode_seconds:.4f} s | "
    f"ratio: {100 * stats_neuralzip.compression_ratio:.2f}%"
)

## 6. Reuse or transfer the setup

A setup can be reused in memory, saved independently, or loaded by a new compressor. For a related target, `map_target()` validates and records the deterministic stream mapping before encoding.

Exact tensor names are preferred. When related architectures represent the same role differently, NeuralZip may use an explicit structural fallback. In this example, TinyLlama stores a separate `lm_head.weight`, while Qwen ties that output head to `model.embed_tokens.weight`; the two BF16 streams reuse the embedding modes and codebook. The mapping report keeps exact and fallback assignments separate.

Shapes and values may differ, and no target histograms are computed. NeuralZip does not search for the closest architecture automatically—the application chooses the source setup. Setup files use Python pickle, so load them only from trusted sources.

In [ ]:
related_model_id = "TinyLlama/TinyLlama-1.1B-Chat-v1.0"
related_model = AutoModelForCausalLM.from_pretrained(
    related_model_id,
    dtype=torch.bfloat16,
).eval()

In [ ]:
# Persist only the reusable clusters and codebooks.
neuralzip.save_setup("qwen-family.setup.pkl", setup=setup)

# Reuse them later without running fit() again.
reused = Compressor("neuralzip", setup="qwen-family.setup.pkl")
mapping = reused.map_target(related_model)
compressed_related, related_stats = reused.encode(
    related_model,
    return_stats=True,
)

print(
    f"Mapped tensors: {mapping['matched_tensors']}/{mapping['target_tensors']} "
    f"(exact: {mapping['exact_tensors']}, "
    f"fallback: {mapping['fallback_tensors']})"
)
print(
    f"Reused NeuralZip — encode: {related_stats.encode_seconds:.4f} s | "
    f"ratio: {100 * related_stats.compression_ratio:.2f}%"
)

## 7. Run inference from compressed GPU weights

`prepare_inference()` does not repeat `fit()` or `encode()`. It converts the already encoded sharing-only payloads from the storage framing to the CUDA decoder layout and attaches them to the executable model.

Selected BF16 linear weights remain compressed on the GPU. Immediately before a transformer block executes, NeuralZip reconstructs that block losslessly into one reusable BF16 workspace. GPU preparation is therefore a separate operation from the `encode_seconds` reported above.

### BF16, NeuralZip, and DFloat11

In [ ]:
# BF16
generator = pipeline(
    "text-generation",
    model=model,
    tokenizer=tokenizer,
    device=model.device,
)
output = generator(
    "Machine learning is",
    max_new_tokens=20,
    do_sample=False,
)
output

In [ ]:
# DFloat11
from dfloat11 import DFloat11Model, compress_model

qwen_block_pattern = {
    r"model\.layers\.\d+": [
        "self_attn.q_proj", "self_attn.k_proj",
        "self_attn.v_proj", "self_attn.o_proj",
        "mlp.gate_proj", "mlp.up_proj", "mlp.down_proj",
    ]
}
dfloat_path = "qwen-dfloat11"

# The official compressor consumes this BF16 source model.
dfloat_source = AutoModelForCausalLM.from_pretrained(
    model_id,
    dtype=torch.bfloat16,
).eval()
compress_model(
    dfloat_source,
    qwen_block_pattern,
    dfloat_path,
    save_single_file=False,
)
del dfloat_source
dfloat_model = DFloat11Model.from_pretrained(
    dfloat_path,
    device="cuda:0",
)

generator = pipeline(
    "text-generation",
    model=dfloat_model,
    tokenizer=tokenizer,
    device=dfloat_model.device,
)
output = generator(
    "Machine learning is",
    max_new_tokens=20,
    do_sample=False,
)
output

In [ ]:
# NeuralZip

# Attach the already encoded payloads to CUDA.
inference_model = compressed_model_neuralzip.prepare_inference(
    model.to("cuda:0")
)

generator = pipeline(
    "text-generation",
    model=inference_model,
    tokenizer=tokenizer,
    device=inference_model.device,
)
output = generator(
    "Machine learning is",
    max_new_tokens=20,
    do_sample=False,
)
output

In [ ]:
# Return the GPU-backed model to ordinary BF16 parameters.
restored_model = inference_model.neuralzip_runtime.restore()

## 8. Decode

Decoding does not need the original setup: the encoded value already carries its payloads and required codebooks. Because NeuralZip is lossless, correctness is checked by comparing every state tensor exactly—including buffers and floating-point bit patterns—rather than with a numerical tolerance.

In [ ]:
def assert_bitwise_equal(reference, recovered):
    """Verify equal state keys, dtypes, shapes, and raw tensor bytes."""
    reference_state = reference.state_dict()
    recovered_state = recovered.state_dict()
    assert reference_state.keys() == recovered_state.keys()

    for name, expected in reference_state.items():
        actual = recovered_state[name]
        expected = expected.detach().cpu().contiguous()
        actual = actual.detach().cpu().contiguous()
        assert expected.dtype == actual.dtype, f"{name}: dtype differs"
        assert expected.shape == actual.shape, f"{name}: shape differs"
        assert torch.equal(
            expected.reshape(-1).view(torch.uint8),
            actual.reshape(-1).view(torch.uint8),
        ), f"{name}: bytes differ"

In [ ]:
# Independently reconstruct a fresh model from NeuralZip's stored payloads.
decoded_neuralzip, decode_stats_neuralzip = neuralzip.decode(
    compressed_model_neuralzip,
    return_stats=True,
)
assert_bitwise_equal(restored_model, decoded_neuralzip)
print(
    f"NeuralZip decode: {decode_stats_neuralzip.decode_seconds:.4f} s "
    "— bitwise exact"
)

## 9. GPU checkpoints and placement

For production-style inference, NeuralZip can save a directly loadable compressed checkpoint rather than attaching an in-memory model:

In [ ]:
from neuralzip import NeuralZipModel

NeuralZipModel.save_pretrained(
    model,
    neuralzip,
    "qwen-neuralzip",
    tokenizer=tokenizer,
)
inference_model = NeuralZipModel.from_pretrained(
    "qwen-neuralzip",
    device_map="auto"
)

The saved directory contains the model configuration, compressed BF16 payloads, and sharded Safetensors for state that remains resident. Loading reconstructs the architecture without first allocating the original compressed weights.

The same loader supports explicit placement, automatic multi-GPU distribution, and compressed-payload CPU offload:

NeuralZip also accepts tensor mappings and ordinary PyTorch modules. Custom module reconstruction can be supplied through `destination=` or `model_factory=` during decoding. Diffusers components and custom GPU block layouts are documented in the repository README.

This notebook intentionally uses the high-level public API. The lower-level codec, CUDA export, archive, and experiment-runner interfaces are implementation and benchmarking tools rather than requirements for normal library use.

In [ ]:
inference_model = NeuralZipModel.from_pretrained(
    "qwen-neuralzip",
    device_map="auto",
    max_memory={0: "22GiB", 1: "22GiB", "cpu": "64GiB"},
    cpu_offload=True,
)